# Data exploration - Strand Bookstore
This notebook contains an exploration of the data shape for used books in the Strand Bookstore's inventory.

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [ ]:
url ="https://www.strandbooks.com/books-and-media.html"

In [ ]:
driver = webdriver.Chrome()
driver.get(url)

wait = WebDriverWait(driver, timeout=30)

old_text = wait.until(lambda driver: driver.find_element(By.CLASS_NAME, "product-item")).text

condition_used_good_btn = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Used - Good')]")))

driver.execute_script("arguments[0].scrollIntoView();", condition_used_good_btn)     

condition_used_good_btn.click()

wait.until(lambda driver: driver.find_element(By.CLASS_NAME, "product-item").text != old_text)

# wait.until(EC.presence_of_element_located((By.XPATH, "//span[contains(text(), 'Active filtering')]")))
html_source = driver.page_source
print(html_source)
driver.quit()

<html lang="en" data-uw-w-loader="" data-uw-rm-lang="false"><head prefix="og: http://ogp.me/ns# category: http://ogp.me/ns/category#">
    <meta charset="utf-8">
<meta name="title" content="Shop Bestselling Books &amp; Media Online - Discover New Reads Today">
<meta name="description" content="Browse top-selling books and media online. Find your next great read, from timeless classics to trending titles. Shop now and expand your library today!">
<meta name="keywords" content="bestselling books, buy books online, books and media, top selling novels, shop books, literature, fiction, non-fiction, trending books, classic books">
<meta name="robots" content="INDEX,FOLLOW">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Shop Bestselling Books &amp; Media Online - Discover New Reads Today</title>
<link rel="preload" as="style" href="/static/version1787201969/frontend/Rave/hyva_child/en_US/css/styles.css">
<link rel="preload" as="style" href="/static/version1787201

In [59]:
soup=BeautifulSoup(html_source,'lxml')
print(soup)

<html data-uw-rm-lang="false" data-uw-w-loader="" lang="en"><head prefix="og: http://ogp.me/ns# category: http://ogp.me/ns/category#">
<meta charset="utf-8"/>
<meta content="Shop Bestselling Books &amp; Media Online - Discover New Reads Today" name="title"/>
<meta content="Browse top-selling books and media online. Find your next great read, from timeless classics to trending titles. Shop now and expand your library today!" name="description"/>
<meta content="bestselling books, buy books online, books and media, top selling novels, shop books, literature, fiction, non-fiction, trending books, classic books" name="keywords"/>
<meta content="INDEX,FOLLOW" name="robots"/>
<meta content="width=device-width, initial-scale=1" name="viewport"/>
<title>Shop Bestselling Books &amp; Media Online - Discover New Reads Today</title>
<link as="style" href="/static/version1787201969/frontend/Rave/hyva_child/en_US/css/styles.css" rel="preload"/>
<link as="style" href="/static/version1787201969/fronten

Each product card has the class name "product-info".

In [65]:
product_list = soup.find_all("div", class_="product-info")
print(product_list)

[<div class="product-info flex flex-col grow">
<!-- Product Name -->
<div class="mb-1.5 items-center justify-center font-semibold text-base text-center">
<a :href="getProductUrl(product)" class="product-item-link hover:text-primary-color" getcategoryname="" index="" x-html="decodeHtmlEntities(product.name)"></a>
</div>
<!-- Author - Only for non-quote items -->
<div class="mb-1.5 items-center justify-center font-normal text-sm text-center" x-show="product.authors &amp;&amp; product.attribute_set_id != 24">
<span x-html="decodeHtmlEntities(truncate(product.authors, 50))"></span>
</div>
<!-- Price -->
<div class="product-item-price pt-1 text-center">
<div class="price-container">
<span class="price" x-text="formatPrice(getProductDisplayPrice(product))"></span>
</div>
</div>
<!-- Action Buttons -->
<template x-if="product.stock_status === 'IN_STOCK'">
<div>
<!-- ===================== Add to Cart ===================== -->
<template x-if="product.attribute_set_id != 24 &amp;&amp; product.__

In [ ]:
data = []
for product in product_list:
    text = product.get_text(separator="\n", strip=True)
    if "Out of Stock" in text:
        continue
    link = product.find("a").get('href')

    title = product.find("a", class_="product-item-link").get_text(strip=True)

    if not title:
        continue

    author = product.find(
        "div",
        class_="mb-1.5 items-center justify-center font-normal text-sm text-center"
    ).get_text(strip=True)

    price = product.find("span", class_="price").get_text(strip=True)

    if product.find("div", class_="out-of-stock"):
        stock_status = "Out of Stock"
    else:
        stock_status = "In Stock"
    product_data = [title, link, author, price, stock_status]
    data.append(product_data)

In [86]:
df = pd.DataFrame(data)
df.head(10)

,0,1,2,3,4
0,Getting to Reparations: How Building a Differe...,https://www.strandbooks.com/getting-to-reparat...,"Brown, Dorothy A.",$18.00,Out of Stock
1,"The Snakes That Ate Florida: Reporting, Essays...",https://www.strandbooks.com/the-snakes-that-at...,"Frazier, Ian",$19.00,Out of Stock
2,Deserving: What the Lives of the Condemned Rev...,https://www.strandbooks.com/deserving-what-the...,"Vartkessian, Elizabeth|Prejean, Sister Helen",$15.00,Out of Stock
3,There Is No Place for Us: Working and Homeless...,https://www.strandbooks.com/there-is-no-place-...,"Goldstone, Brian",$12.00,Out of Stock
4,The House of Beauty: Lessons from the Image In...,https://www.strandbooks.com/the-house-of-beaut...,"Sicardi, Arabelle",$18.00,Out of Stock
5,The First Shots: The Epic Rivalries and Heroic...,https://www.strandbooks.com/first-shots-the-ep...,"Borrell, Brendan",$10.00,Out of Stock
6,The Last Supper: How to Overcome the Coming Fo...,https://www.strandbooks.com/last-supper-how-to...,"Kass, Sam",$24.00,Out of Stock
7,Anointed: The Extraordinary Effects of Social ...,https://www.strandbooks.com/anointed-the-extra...,"Stuart, Toby",$17.50,Out of Stock
8,Mexico's Day of the Dead: A Celebration of Lif...,https://www.strandbooks.com/mexico-s-day-of-th...,"Navarro, Luisa|Chitnis, Christine",$28.00,Out of Stock
9,"The Afghans: Three Lives Through War, Love, an...",https://www.strandbooks.com/the-afghans-three-...,"Seierstad, Åsne",$15.00,Out of Stock
